Reset the Demo Table

In [0]:
DROP TABLE IF EXISTS workspace.default.scd1_customers;
DROP TABLE IF EXISTS workspace.default.scd2_customers;
DROP TABLE IF EXISTS workspace.default.customer_updates;

In [0]:
--Create the SCD Type 1 table
CREATE TABLE workspace.default.scd1_customers(
	customer_id INT,
	customer_name STRING,
	city STRING
)
USING DELTA;

In [0]:
--Insert the initial customer records
INSERT INTO workspace.default.scd1_customers VALUES
(101,'Arun','Chennai'),
(102,'Priya','Bangalor'),
(103,'John','Mumbai');

In [0]:
SELECT * FROM workspace.default.scd1_customers;

In [0]:
CREATE OR REPLACE TABLE workspace.default.customer_updates(
    customer_id INT,
    customer_name STRING,
    city STRING
)
USING DELTA;


In [0]:
INSERT INTO workspace.default.customer_updates VALUES
(101,'Arun','Hydrabad'),
(104,'David','Pune');

In [0]:
SELECT * FROM workspace.default.customer_updates;

#### Implement SCD Type1 using MERGE INTO

In [0]:
MERGE INTO workspace.default.scd1_customers AS target
USING workspace.default.customer_updates AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN 
UPDATE
    SET target.customer_name = source.customer_name,
        target.city = source.city
WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    city
)
VALUES(
    source.customer_id,
    source.customer_name,
    source.city
);
SELECT * FROM workspace.default.scd1_customers;

In [0]:
--Create the SCD Type2 table

CREATE TABLE workspace.default.scd2_customers(
	customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
	customer_id INT,
	customer_name STRING,
	city STRING,
	effective_start_date DATE,
	effective_end_date DATE,
	is_current BOOLEAN
)
USING DELTA;

In [0]:
--Insert the initial records
INSERT INTO workspace.default.scd2_customers(
	customer_id,
	customer_name,
	city,
	effective_start_date,
	effective_end_date,
	is_current
)
VALUES
(101,'Arun','Chennai',DATE '2026-11-01',NULL,true),
(102,'Priya','Bangalore',DATE '2026-11-01',NULL,true),
(103,'John','Mumbai',DATE '2026-11-01',NULL,true);

In [0]:
SELECT * FROM workspace.default.scd2_customers;

#### Prepare the staged source

In [0]:
CREATE OR REPLACE TEMP VIEW  scd2_staged_updates AS

--New version for customers whose attributes changed
SELECT 
   NULL AS merge_key,
   source.customer_id,
   source.customer_name,
   source.city 
FROM workspace.default.customer_updates AS source
JOIN workspace.default.scd2_customers AS target
ON source.customer_id = target.customer_id
AND target.is_current = TRUE

WHERE NOT (source.customer_name <=> target.customer_name)
OR NOT (source.city <=> target.city)

UNION ALL

--Source rows used to match existing records or insert new rows
SELECT 
   source.customer_id AS merge_key,
   source.customer_id,
   source.customer_name,
   source.city 
FROM workspace.default.customer_updates AS source
;

In [0]:
--verify temp view
SELECT * FROM scd2_staged_updates;

#### Execute the SCD Type2

In [0]:
MERGE INTO workspace.default.scd2_customers AS target
USING scd2_staged_updates AS source
ON target.customer_id = source.merge_key
AND target.is_current = TRUE

WHEN MATCHED AND (
    NOT (target.customer_name <=> source.customer_name)
    OR NOT (target.city <=> source.city)
)  THEN
UPDATE
    SET target.effective_end_date = current_date(),
        target.is_current = FALSE

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    city,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES(
    source.customer_id,
    source.customer_name,
    source.city,
    current_date(),
    NULL,
    TRUE
);

In [0]:
--Verify scd2 results

SELECT * FROM workspace.default.scd2_customers
ORDER BY customer_id,effective_start_date;

#### Retrive only the current customer records

In [0]:
SELECT customer_id,
      customer_name,
      city 
FROM workspace.default.scd2_customers
WHERE is_current = TRUE
ORDER BY customer_id;

#### Retrive customer history

In [0]:
SELECT customer_id,
      customer_name,
      city,
      effective_start_date,
      effective_end_date,
      is_current
FROM workspace.default.scd2_customers 
WHERE customer_id=101 
ORDER BY effective_start_date;